In [49]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [53]:
# define variables
num_classes = 10
batch_size = 64
num_channels = 1 # grayscale
img_size = 28
patch_size = 7 # 4
num_patches = (img_size // patch_size) ** 2
embedding_size = 64 # 1 patch -> 1 vector
mlp_hidden_nodes = 128
attention_heads = 4
transformer_blocks = 4
learning_rate = 0.001
epochs = 5

In [55]:
# 1. Patch Embedding
class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding = nn.Conv2d(num_channels, embedding_size, kernel_size=patch_size, stride=patch_size) # no trainable params

  def forward(self, x):
    # x.shape is (64, 1, 28, 28) (batch, channels, height, width)
    x = self.patch_embedding(x) # (64, 64, 4, 4) (batch, embedding_size/channels, patch_rows, patch_cols)
    # flatten the last 2 dims (4 x 4)
    x = x.flatten(2) # (64, 64, 16)
    x = x.transpose(1, 2) # (64, 16, 64) (64 images, 16 patches per image, 64-dim embedding for each patch)
    return x
    # [CLASS] and positional embedding

In [57]:
# 2. Encoder
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm_1 = nn.LayerNorm(embedding_size)
    self.layer_norm_2 = nn.LayerNorm(embedding_size)
    self.multi_head_attn = nn.MultiheadAttention(embed_dim=embedding_size, num_heads=attention_heads, batch_first=True)
    self.mlp = nn.Sequential(
        nn.Linear(embedding_size, mlp_hidden_nodes),
        nn.GELU(),
        nn.Linear(mlp_hidden_nodes, embedding_size)
    )

  def forward(self, x):
    residual_1 = x
    x = self.layer_norm_1(x)
    x = self.multi_head_attn(x, x, x)[0]
    x = x + residual_1

    residual_2 = x
    x = self.layer_norm_2(x)
    x = self.mlp(x)
    x = x + residual_2

    return x

In [58]:
# 3. MLP Head
class MLPHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm_1 = nn.LayerNorm(embedding_size)
    self.mlp_head = nn.Linear(embedding_size, num_classes)

  def forward(self, x):
    x = self.layer_norm_1(x)
    x = self.mlp_head(x)
    return x

In [59]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding = PatchEmbedding()
    self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_size)) # (1 batch, 1 cls token, embedding_size)
    self.position_embedding = nn.Parameter(torch.randn(1, num_patches+1, embedding_size))
    self.transformer_blocks = nn.Sequential(*[Encoder() for _ in range(transformer_blocks)])
    self.mlp_head = MLPHead()

  def forward(self, x):
    # x: (batch_size, num_channels, img_size, img_size) example: (64, 1, 28, 28)
    x = self.patch_embedding(x) # (batch_size, num_patches, embedding_size) (64, 16, 64)
    B = x.size(0) # current batch size
    class_tokens = self.cls_token.expand(B, -1, -1)
    x = torch.cat((class_tokens, x), dim=1)
    x = x + self.position_embedding
    x = self.transformer_blocks(x)
    x = x[:, 0] # pick the cls token from every batch
    x = self.mlp_head(x)
    return x